In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

In [84]:
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f+extra)
    return(prob.value,q.value,q_b.value)

In [4]:
def dual (sets,p,R,r,m,r_f,a):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R.dot(a))[i]-(1-sum(a))*r_f - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    obj= cp.Minimize(alpha + beta + gamma * (r-1) + z4 + z2)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,v.value,lbda.value,alpha.value,beta.value,gamma.value,t.value)

In [101]:
np.random.seed(10)
N=7
x=np.arange(1,N)
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]

In [102]:
p = np.random.rand(N)
p = p/sum(p)
#p = np.zeros(N)+1/N
R = np.random.rand(N,1)*2-1
sets =psets

In [103]:
print(p)
print(R)

[0.24914333 0.00670306 0.20467393 0.24187022 0.16102213 0.07261129
 0.06397604]
[[ 0.52106142]
 [-0.66177833]
 [-0.82332037]
 [ 0.37071964]
 [ 0.90678669]
 [-0.99210347]
 [ 0.02438453]]


In [112]:
a = np.array([0.6])
r = 0
m = 0.99
r_f = 0.001
robustcheck(a,R,r,p,m,r_f)

0.5948619471097936


(1.1393339625437173,
 array([0.24915419, 0.00670229, 0.20466087, 0.24188223, 0.1610048 ,
        0.07261767, 0.06397796]),
 array([3.75674765e-08, 9.98251941e-08, 1.14422667e-07, 2.55210365e-08,
        1.11022302e-16, 9.99999669e-01, 2.78776974e-08]))

0.5948620381641259


(1.1393340535980496,
 array([0.24913972, 0.00670135, 0.20464633, 0.24189762, 0.16102498,
        0.07261072, 0.06397928]),
 array([1.36060318e-08, 6.40484245e-09, 7.31031113e-08, 6.88464102e-09,
        1.11022302e-16, 9.99999885e-01, 3.59885057e-09]))

In [80]:
dual (sets,p,R,r,m,r_f,a)

(0.8233203712823867,
 array([[-4.85900015e-11, -5.08273252e-11, -4.99037147e-11,
         -5.08016132e-11],
        [-5.06601201e-11, -4.00770563e-11, -0.00000000e+00,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00, -4.88070569e-11,
         -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00,
         -4.97513453e-11],
        [-4.56207618e-11, -4.58189950e-11, -0.00000000e+00,
         -0.00000000e+00],
        [-4.54629898e-11, -0.00000000e+00, -4.45116437e-11,
         -0.00000000e+00],
        [-4.54361154e-11, -0.00000000e+00, -0.00000000e+00,
         -4.56380042e-11],
        [-0.00000000e+00, -4.56740133e-11, -4.44988595e-11,
         -0.00000000e+00],
        [-0.00000000e+00, -4.56275880e-11, -0.00000000e+00,
         -4.56101813e-11],
        [-0.00000000e+00, -0.00000000e+00, -4.47433728e-11,
         -4.58743639e-11],
        [-4.74381588e-11, -4.76255202e-11, -4.65610347e-11,
         -0.00000000e+00],
        [-4.74000374e-

In [54]:
[probv,vv,lbdav,alphav,betav,gammav,tv]=dual (sets,p,R,r,m,r_f,a)
N = len(p)
M = len(sets)
cons1 =np.zeros(N)
cons2 = np.zeros(N)
z0 = 0
for j in range(M):
    z9 = -np.min(vv[j,sets[j]])*(1-m)+lbdav[j]
    z0 = z0 + max(z9,0)
for i in range(N):
    lbdsom = 0
    for j in range(M):
        if i in sets[j]:
            lbdsom = lbdsom + lbdav[j]
    cons1[i] = R.dot(a)[i] + betav + lbdsom
    cons2[i] = gammav * np.exp((-alphav+sum(vv[0:M:1,i]))/gammav)-tv[i]
print(cons1)
print(cons2)
print(-1+alphav+betav+gammav*r+sum(p*tv)+z0)

[ 5.70003067e-09 -5.38403810e-09  6.53539289e+00  1.70622106e+01]
[-8.10329546e-08 -2.98001260e-06 -3.94966149e-08 -2.06583066e-08]
[24.65265508]


In [62]:
sum(p)

1.0